# Discriminative latent spaces for the private-encoder dataset MoE

This notebook extends notebook 16 without mixing several questions into one result. It keeps the same full-private-encoder MoE topology, dense task-driven gate, four NF-v3 datasets, widths, optimizer, and A/B/C schedule. The intervention is the Stage-A representation objective: the default is **CE + imbalance-aware balanced supervised contrastive loss** applied directly to the 64-dimensional latent vector consumed by the MoE.

After training, the notebook evaluates downstream detection and inspects the representation at every meaningful point: pooled Stage A, the unchanged Stage-B gate encoder, every private Stage-B expert encoder, the Stage-C gate encoder, and every private Stage-C expert encoder. Geometry is measured in the original 64-D space; PCA and UMAP are diagnostic views only.

The primary comparison is this run versus notebook 16's CE private-encoder run on the same signed split and seed. Change one representation objective at a time and use validation metrics to choose a method; do not choose a winner from test-set plots.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = "selimsidan"
GITHUB_REPO = "dataset_moe_nids"
GITHUB_BRANCH = "main"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"

DRIVE_DATA_DIR = "/content/drive/MyDrive/NIDS_datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs"
EXECUTION_MODE = "out_of_core_full"  # out_of_core_full | in_memory_smoke

ACTIVE_DATASETS = [
    "NF-UNSW-NB15-v3",
    "NF-ToN-IoT-v3",
    "NF-BoT-IoT-v3",
    "NF-CICIDS2018-v3",
]
ARCHITECTURE = "moe_dataset_private_encoders"
SEED = 0
RUN_NAME = "nfv3_4way_private_balanced_supcon_seed0_v1"
REFERENCE_PRIVATE_RUN = "nfv3_4way_moe_private_encoders_seed0_v1"  # notebook 16 CE control

REPRESENTATION_OBJECTIVE = "balanced_supcon"  # ce | supcon | balanced_supcon | center | arcface
REPRESENTATION_SAMPLING = "class_domain_balanced"  # legacy | class_domain_balanced
REPRESENTATION_CLASS_WEIGHTING = "none"  # keep fixed across balanced CE and metric-loss candidates
REPRESENTATION_WEIGHT = 0.10
SUPCON_TEMPERATURE = 0.10
CENTER_WEIGHT = 0.01
ARCFACE_MARGIN = 0.30
ARCFACE_SCALE = 30.0
MIN_PER_CLASS_PER_BATCH = 4

LATENT_EVAL_SPLIT = "val"  # use val while choosing; change to test only after locking the method
FOCUS_CLASSES = None  # None = every class with held-out support; or provide a shorter list
RUN_UMAP = True
RUN_TESTS = True
FORCE_RESTART = False

LATENT_DIM = 64
ENCODER_HIDDEN_DIMS = [128]
EXPERT_HIDDEN_DIMS = [45]
DROPOUT = 0.2
EPOCHS_A = 30
EPOCHS_B = 30
EPOCHS_C = 30
BATCH_SIZE = 512
GATE_SUPERVISION = "none"
EXPERT_UPDATE_POLICY = "all"
LAMBDA_BALANCE = 0.1
LAMBDA_EXPERT_ANCHOR = 0.0
# ==================================================================

## 1. Secure checkout and environment setup

Use a GPU runtime. The GitHub token is passed through a temporary HTTP header and is not stored in the clone URL or Git configuration.

In [ ]:
import base64, os, subprocess, sys, time
from pathlib import Path
try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("This notebook is intended for Google Colab.") from exc
drive.mount("/content/drive")
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f"Add {GITHUB_SECRET_NAME} in Colab Secrets and grant notebook access.")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ | {
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
}
repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
repo_dir = Path("/content") / GITHUB_REPO
if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", repo_url, str(repo_dir)], env=git_env, check=True)
git_env.clear(); token = auth = None
os.chdir(repo_dir)
os.environ["NIDS_DRIVE_BASE"] = DRIVE_DATA_DIR
os.environ["NIDS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
os.environ["NIDS_SCRATCH_DIR"] = "/content/dataset_moe_nids_scratch"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Repository:", repo_dir)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

## 2. Validate the scientific contract

This run deliberately trains a new Stage A because reusing notebook 16's CE checkpoint would remove the intervention. Everything downstream remains paired with notebook 16. The support table also makes classes absent from a dataset explicit instead of assigning them artificial zero-quality clusters.

In [ ]:
import torch
from data.registry import get_spec
from training.config import load_config

valid_objectives = {"ce", "supcon", "balanced_supcon", "center", "arcface"}
if REPRESENTATION_OBJECTIVE not in valid_objectives:
    raise ValueError(f"Unknown objective: {REPRESENTATION_OBJECTIVE}")
if REPRESENTATION_OBJECTIVE in {"supcon", "balanced_supcon"} and REPRESENTATION_SAMPLING != "class_domain_balanced":
    raise ValueError("Contrastive runs must use class_domain_balanced sampling so rare classes have positives.")
if EXECUTION_MODE not in {"out_of_core_full", "in_memory_smoke"}:
    raise ValueError("Unknown EXECUTION_MODE")
if ARCHITECTURE != "moe_dataset_private_encoders":
    raise ValueError("Notebook 17 targets the private-encoder MoE from notebook 16.")
if GATE_SUPERVISION != "none" or EXPERT_UPDATE_POLICY != "all":
    raise ValueError("Keep the Stage-C contract paired with notebook 16.")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU.")
missing = {}
for name in ACTIVE_DATASETS:
    spec = get_spec(name)
    if not any(Path(path).is_file() for path in spec.paths):
        missing[name] = spec.paths
if missing:
    raise FileNotFoundError("Missing datasets:\n" + "\n".join(f"  {name}: {paths}" for name, paths in missing.items()))
aliases = [get_spec(name).feature_alias for name in ACTIVE_DATASETS]
if any(value != aliases[0] for value in aliases[1:]) or len(aliases[0]) != 47:
    raise ValueError("Full mode requires the same four schema-compatible 47-feature NF-v3 datasets.")
print("GPU:", torch.cuda.get_device_name(0))
print("Objective:", REPRESENTATION_OBJECTIVE, "sampling:", REPRESENTATION_SAMPLING)
print("Run:", RUN_NAME)

In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
else:
    print("Tests skipped by configuration.")

## 3. Train and persist latent reports stage by stage

The balanced sampler guarantees multiple examples for every present class and draws them across datasets whenever that class occurs in more than one source. After each completed training stage, its latent metrics, embeddings, manifests, PCA/UMAP views, and per-class figures are written atomically to Drive before the next stage starts. The cumulative directory is refreshed after every stage, so an interrupted Stage B or C cannot erase the completed Stage-A analysis.

In [ ]:
effective_run_name = RUN_NAME if EXECUTION_MODE == "out_of_core_full" else RUN_NAME + "_smoke"
epochs_a = 1 if EXECUTION_MODE == "in_memory_smoke" else EPOCHS_A
epochs_b = 1 if EXECUTION_MODE == "in_memory_smoke" else EPOCHS_B
epochs_c = 1 if EXECUTION_MODE == "in_memory_smoke" else EPOCHS_C
if EXECUTION_MODE != "out_of_core_full":
    raise RuntimeError("Incremental private-encoder latent reporting requires EXECUTION_MODE='out_of_core_full'.")
base_overrides = [
    f"run_name={effective_run_name}", f"seed={SEED}", f"data.split_seed={SEED}",
    f"architecture={ARCHITECTURE}",
    "data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]",
    "training.device=cuda",
    f"training.epochs_a={epochs_a}", f"training.epochs_b={epochs_b}", f"training.epochs_c={epochs_c}",
    f"training.batch_size={BATCH_SIZE}", f"training.min_per_class_per_batch={MIN_PER_CLASS_PER_BATCH}",
    "training.selection_mode=fixed_epochs",
    f"training.representation.objective={REPRESENTATION_OBJECTIVE}",
    f"training.representation.sampling={REPRESENTATION_SAMPLING}",
    f"training.representation.class_weighting={REPRESENTATION_CLASS_WEIGHTING}",
    f"training.representation.weight={REPRESENTATION_WEIGHT}",
    f"training.representation.temperature={SUPCON_TEMPERATURE}",
    f"training.representation.center_weight={CENTER_WEIGHT}",
    f"training.representation.arc_margin={ARCFACE_MARGIN}",
    f"training.representation.arc_scale={ARCFACE_SCALE}",
    f"model.latent_dim={LATENT_DIM}",
    "model.encoder.hidden_dims=[" + ",".join(map(str, ENCODER_HIDDEN_DIMS)) + "]",
    "model.expert.hidden_dims=[" + ",".join(map(str, EXPERT_HIDDEN_DIMS)) + "]",
    f"model.encoder.dropout={DROPOUT}", f"model.expert.dropout={DROPOUT}",
    "model.gate.routing=dense", "training.stage_c_unfreeze=all",
    f"training.stage_c.gate_supervision={GATE_SUPERVISION}",
    f"training.stage_c.expert_update_policy={EXPERT_UPDATE_POLICY}",
    f"training.stage_c.lambda_expert_anchor={LAMBDA_EXPERT_ANCHOR}",
    f"load_balance.lambda_balance={LAMBDA_BALANCE}",
]

import pandas as pd
import queue, shutil, threading
from collections import deque
from IPython.display import display, Image
from training.out_of_core_data import prepare_out_of_core_data
from evaluation.latent_space import (
    combine_latent_reports, evaluate_private_latent_checkpoints,
    plot_class_focus, plot_latent_snapshots,
)

result_dir = Path(DRIVE_OUTPUT_DIR) / "results" / effective_run_name
latent_root = result_dir / "latent"
cumulative_dir = latent_root / "cumulative"
config = load_config("config/default.yaml", base_overrides)
context = prepare_out_of_core_data(config)  # reused for each checkpoint report

def run_streaming(command, env, heartbeat_seconds=30):
    process = subprocess.Popen(
        command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    output = queue.Queue()
    def pump_output():
        try:
            for line in process.stdout:
                output.put(line)
        finally:
            output.put(None)
    threading.Thread(target=pump_output, daemon=True).start()
    print(f"[notebook] child PID={process.pid}; streaming combined stdout/stderr", flush=True)
    started = time.monotonic()
    try:
        while True:
            try:
                line = output.get(timeout=heartbeat_seconds)
            except queue.Empty:
                status = subprocess.run(
                    ["ps", "-o", "etime=,%cpu=,%mem=,rss=,stat=", "-p", str(process.pid)],
                    text=True, capture_output=True, check=False,
                ).stdout.strip()
                print(
                    f"[notebook] heartbeat after {(time.monotonic() - started) / 60:.1f} min; "
                    f"PID={process.pid}; etime/cpu%/mem%/rssKB/state={status or 'process exiting'}",
                    flush=True,
                )
                continue
            if line is None:
                break
            print(line, end="", flush=True)
    except KeyboardInterrupt:
        print(f"[notebook] interrupting child PID={process.pid}", flush=True)
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
        raise
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def launch_training(stages, *, final_evaluation, force_restart=False):
    stage_value = "[" + ",".join(stages) + "]"
    run_overrides = [
        *base_overrides, f"training.stages={stage_value}",
        f"training.run_final_evaluation={str(final_evaluation).lower()}",
        f"training.force_restart={str(force_restart).lower()}",
    ]
    cmd = [sys.executable, "-u", "-m", "training.ooc_run", "--config", "config/default.yaml"]
    for override in run_overrides:
        cmd += ["--set", str(override)]
    run_env = os.environ.copy()
    run_env["PYTHONUNBUFFERED"] = "1"
    checkpoint_dir = Path(DRIVE_OUTPUT_DIR) / "checkpoints" / effective_run_name
    log_path = checkpoint_dir / "train.log"
    print("Launching unbuffered:", " ".join(cmd), flush=True)
    print("Persistent log:", log_path, flush=True)
    started = time.monotonic()
    try:
        run_streaming(cmd, run_env)
    except subprocess.CalledProcessError as exc:
        elapsed = (time.monotonic() - started) / 60
        print(f"Training FAILED after {elapsed:.1f} min with exit code {exc.returncode}.", flush=True)
        for label, path in (("Colab local", "/content"), ("Drive output", DRIVE_OUTPUT_DIR)):
            try:
                usage = shutil.disk_usage(path)
                print(f"{label}: {usage.free / 2**30:.1f} GiB free / {usage.total / 2**30:.1f} GiB total")
            except OSError as storage_exc:
                print(f"{label}: storage information unavailable: {storage_exc}")
        if log_path.is_file():
            with log_path.open(errors="replace") as handle:
                print("--- last 80 persistent log lines ---")
                print("".join(deque(handle, maxlen=80)))
        raise
    print(f"Completed stages={stages} in {(time.monotonic()-started)/3600:.2f} hours.")

def save_plots(report, output_dir):
    output_dir = Path(output_dir)
    pca = plot_latent_snapshots(report, str(output_dir), method="pca", random_seed=SEED)
    projections = plot_latent_snapshots(
        report, str(output_dir), method="umap", random_seed=SEED
    ) if RUN_UMAP else pca
    available = sorted(set(report["manifest"]["class"]))
    requested = available if FOCUS_CLASSES is None else list(FOCUS_CLASSES)
    for class_name in requested:
        if class_name not in available:
            print(f"Skipping {class_name}: no {LATENT_EVAL_SPLIT} support.")
            continue
        path = plot_class_focus(report, projections, class_name, str(output_dir))
        print("Saved class focus:", class_name, path)
        if FOCUS_CLASSES is not None:
            display(Image(filename=path))

stage_reports = {}
for index, stage in enumerate(("A", "B", "C")):
    launch_training([stage], final_evaluation=False, force_restart=FORCE_RESTART and index == 0)
    stage_dir = latent_root / f"stage_{stage}"
    reference = stage_reports.get("A") if stage == "B" else None
    stage_reports[stage] = evaluate_private_latent_checkpoints(
        config, context, split_name=LATENT_EVAL_SPLIT, output_dir=str(stage_dir),
        device="cuda", stages=[stage],
        sample_manifest=str(latent_root / "stage_A") if stage != "A" else None,
        reference_report=reference, reuse_completed=True,
    )
    save_plots(stage_reports[stage], stage_dir)
    latent_report = combine_latent_reports(stage_reports.values(), cumulative_dir)
    save_plots(latent_report, cumulative_dir)
    print(f"Stage {stage} latent bundle safely persisted to {stage_dir}")

# Only now run the expensive final downstream report. Empty stages means
# load the completed checkpoints without retraining anything.
launch_training([], final_evaluation=True, force_restart=False)
overrides = base_overrides
print("Final downstream evaluation and every latent bundle are complete.")

## 4. Downstream comparison against notebook 16

Macro-F1 is the primary metric. Macro PR-AUC is especially useful under imbalance; older notebook-16 result files may not contain it, so the comparison uses every common metric and still displays the new absolute values.

In [ ]:
import pandas as pd
from IPython.display import display, Image

result_dir = Path(DRIVE_OUTPUT_DIR) / "results" / effective_run_name
reference_dir = Path(DRIVE_OUTPUT_DIR) / "results" / REFERENCE_PRIVATE_RUN
candidate = pd.read_csv(result_dir / "Overall_Metrics.csv")
display(candidate)
reference_path = reference_dir / "Overall_Metrics.csv"
if reference_path.is_file():
    reference = pd.read_csv(reference_path)
    preferred = ["accuracy", "balanced_accuracy", "macro_precision", "macro_recall", "macro_f1", "pr_auc_ovr_macro", "roc_auc_ovr_macro"]
    metrics = [name for name in preferred if name in candidate.columns and name in reference.columns]
    comparison = candidate[["origin", *metrics]].set_index("origin").add_prefix("candidate__").join(
        reference[["origin", *metrics]].set_index("origin").add_prefix("reference__"), how="inner")
    for metric in metrics:
        comparison[f"delta__{metric}"] = comparison[f"candidate__{metric}"] - comparison[f"reference__{metric}"]
    display(comparison.reset_index())
else:
    comparison = None
    print("Notebook-16 reference not found:", reference_path)
display(pd.read_csv(result_dir / "Per_Class_Metrics.csv").query("origin == 'ALL' and is_native_class == True").sort_values("f1"))
display(pd.read_csv(result_dir / "Per_Dataset_Metrics.csv").sort_values("macro_f1"))

## 5. Inspect the saved cumulative latent report

These artifacts were already saved after their corresponding training stages. Quantitative metrics use normalized 64-D embeddings: silhouette, within-class radius, nearest-rival centroid distance, centroid margin, cross-dataset centroid dispersion, alignment/uniformity, effective rank, k-NN, linear class probe, and linear dataset probe. A high dataset-probe score is not automatically bad here—the gate may need dataset signal—but it reveals the trade-off between class invariance and expert routing.

In [ ]:
print("=== Snapshot geometry ===")
display(latent_report["snapshots"].sort_values(["stage", "encoder"]))
print("=== Frozen probes ===")
display(latent_report["probes"].sort_values(["target", "probe", "stage", "encoder"]))
print("=== Per-class geometry ===")
display(latent_report["per_class"].sort_values(["class", "stage", "encoder"]))
support = pd.crosstab(latent_report["manifest"]["class"], latent_report["manifest"]["dataset"])
support = support.reindex(config["data"]["active_classes"], fill_value=0)
display(support.style.background_gradient(cmap="Blues"))

## 6. Saved PCA/UMAP and per-class inspection

Every snapshot uses the same persisted sample manifest and class colors. Separate private encoders can rotate their coordinate systems arbitrarily, so compare their quantitative geometry and class neighborhoods—not the absolute x/y location of a cluster between panels. Set `FOCUS_CLASSES` to a list only when you want a smaller set displayed inline; `None` still saves a dedicated figure for every supported class.

In [ ]:
print("Stage-specific latent artifacts:")
for stage in ("A", "B", "C"):
    print(f"  Stage {stage}:", latent_root / f"stage_{stage}")
print("Cumulative metrics and cross-stage figures:", cumulative_dir)
print("Classes with dedicated focus figures:", sorted(set(latent_report["manifest"]["class"])))

## Interpretation and next run

Prefer this representation objective only when validation macro-F1 improves over notebook 16 and the gain is not confined to one dataset. Use per-class recall/PR-AUC to identify who benefits, then use centroid margin, silhouette, and probe recall to explain why. Check the dataset probe and gate/expert reports before concluding that stronger class clustering is universally helpful: removing all dataset information can damage a dataset-expert router.

The Stage-B gate row is intentionally marked equivalent to Stage A because that encoder is not trained in Stage B. The four `B__expert::*` rows are the genuine per-expert representations. Stage C then shows whether joint MoE optimization preserves or erodes the Stage-A geometry. Unsupported classes are reported through the support table and skipped—not assigned a misleading zero score.

For the focused search, keep the split and seed fixed and create separate run names for: balanced CE control, SupCon weights 0.05/0.10/0.20, balanced-SupCon weights 0.05/0.10/0.20, center weights 0.001/0.01/0.1, and ArcFace margins 0.1/0.3/0.5. Choose on validation macro-F1, lock the winner, then repeat the winner and CE reference across seeds 0–4 before inspecting test results.